Random Forest + SMOTE

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

from sklearn.ensemble import Random Forest + SMOTEClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    recall_score, precision_score, f1_score
)

# imbalanced-learn for SMOTE
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    print('[INFO] imbalanced-learn not installed. ')
    print('       Run: pip install imbalanced-learn')
    print('       SMOTE comparison will be skipped.')

matplotlib.rcParams['figure.facecolor'] = 'white'
matplotlib.rcParams['axes.facecolor']   = 'white'

DANGER  = '#E74C3C'
WARN    = '#F39C12'
SAFE    = '#27AE60'
NAVY    = '#2C3E50'
ACCENT  = '#2980B9'

BASE         = Path('..').resolve()
FIGURES_DIR  = BASE / 'reports' / 'figures'
ML_DIR       = BASE / 'ml'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ML_DIR.mkdir(parents=True, exist_ok=True)

print('All imports OK')
print(f'Figures → {FIGURES_DIR}')
print(f'Model   → {ML_DIR}')

---
## 1. Load Data & Create Target

`delivery_failure` = `delivery_failed` — the `DELIVERY_ATTEMPTED` scan status from the LMRC dataset. Excluding `delivery_failed` as a feature prevents data leakage.

In [ ]:
df = pd.read_csv(BASE / 'data' / 'packages_validation.csv')

# ── Target: delivery_failure = delivery_failed ──────────────────────
df['delivery_failure'] = df['delivery_failed']

print(f'Dataset shape   : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Failure count   : {df["delivery_failure"].sum()} ({df["delivery_failure"].mean():.2%})')
print(f'Class ratio     : ~{int(df["delivery_failure"].value_counts()[0] / df["delivery_failure"].sum())}:1')
print()
print('Column dtypes:')
print(df.dtypes)
df.head(3)

---
## 2. Feature Engineering

Steps:
- **Drop zero-variance columns**: `days_in_fc`, `weather_risk` (identical value for all rows)
- **Drop data leakage column**: `delivery_failed` (IS the target)
- **Encode categoricals**: carrier, shift, package_type → label-encoded integers
- **Bucket route distance**: < 40 km / 40–60 km / > 60 km

In [ ]:
# ── Zero-variance audit ──────────────────────────────────────────────────
print('Zero-variance check:')
for col in ['days_in_fc', 'weather_risk']:
    print(f'  {col}: {df[col].nunique()} unique value(s) → {df[col].unique()}')

# ── Drop zero-variance + leakage columns ─────────────────────────────────
EXCLUDE = ['package_id', 'days_in_fc', 'weather_risk', 'delivery_failed',
           'delivery_failure']

# ── Encode categoricals ───────────────────────────────────────────────────
encoders = {}
df_enc = df.copy()

for col in ['carrier', 'shift', 'package_type']:
    le = LabelEncoder()
    df_enc[col + '_enc'] = le.fit_transform(df_enc[col])
    encoders[col] = le
    print(f'{col} classes: {list(le.classes_)}')

# ── Distance bucket ───────────────────────────────────────────────────────
df_enc['dist_bucket'] = pd.cut(
    df_enc['route_distance_km'],
    bins=[-1, 40, 60, 9999],
    labels=[0, 1, 2]   # 0=<40km, 1=40-60km, 2=>60km
).astype(int)

FEATURES = [
    'carrier_enc', 'shift_enc', 'package_type_enc',
    'dist_bucket', 'packages_in_route',
    'double_scan', 'short_service_time', 'cr_number_missing'
]

X = df_enc[FEATURES].values
y = df_enc['delivery_failure'].values

print(f'\nFeature matrix: {X.shape}')
print(f'Target vector : {y.shape}  ({y.sum()} positives)')
print(f'Features used : {FEATURES}')

---
## 3. Train / Test Split

Stratified split preserves the 0.7% failure rate in both subsets. With only 25 failures, an 80/20 split gives ~20 failures for training and ~5 for testing — a very small but real test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Train / Test split (stratified):')
print(f'  Train: {len(X_train):,} rows | {y_train.sum()} failures ({y_train.mean():.2%})')
print(f'  Test : {len(X_test):,} rows  | {y_test.sum()} failures ({y_test.mean():.2%})')

# Confirm class ratio preserved
print(f'\nClass ratio (train): {int((1-y_train.mean())/y_train.mean())}:1')
print(f'Class ratio (test) : {int((1-y_test.mean())/y_test.mean())}:1')

Random Forest + SMOTE

In [ ]:
rf_base = Random Forest + SMOTEClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    max_depth=8,
    min_samples_leaf=3,
)
rf_base.fit(X_train, y_train)

y_pred_base  = rf_base.predict(X_test)
y_prob_base  = rf_base.predict_proba(X_test)[:, 1]

auc_base     = roc_auc_score(y_test, y_prob_base)
recall_base  = recall_score(y_test, y_pred_base, zero_division=0)
prec_base    = precision_score(y_test, y_pred_base, zero_division=0)
ap_base      = average_precision_score(y_test, y_prob_base)

print('=== Baseline RF (class_weight=balanced) ===')
print(f'AUC-ROC          : {auc_base:.4f}')
print(f'Avg Precision    : {ap_base:.4f}')
print(f'Recall  (fail)   : {recall_base:.4f}')
print(f'Precision (fail) : {prec_base:.4f}')
print()
print(classification_report(y_test, y_pred_base,
      target_names=['Delivered', 'Failed'], digits=4, zero_division=0))

---
## 5. SMOTE Comparison

SMOTE (Synthetic Minority Oversampling) generates synthetic failure samples in feature space to balance the training set. We compare this against the `class_weight='balanced'` baseline to decide which performs better.

In [ ]:
if HAS_SMOTE:
    # SMOTE on training set only — NEVER on test set
    smote = SMOTE(random_state=42, k_neighbors=min(5, y_train.sum() - 1))
    X_sm, y_sm = smote.fit_resample(X_train, y_train)
    print(f'Before SMOTE: {len(X_train)} rows | {y_train.sum()} failures')
    print(f'After  SMOTE: {len(X_sm)} rows | {y_sm.sum()} failures')

    rf_smote = Random Forest + SMOTEClassifier(
        n_estimators=300, random_state=42, n_jobs=-1,
        max_depth=8, min_samples_leaf=3
    )
    rf_smote.fit(X_sm, y_sm)

    y_pred_sm  = rf_smote.predict(X_test)
    y_prob_sm  = rf_smote.predict_proba(X_test)[:, 1]

    auc_sm    = roc_auc_score(y_test, y_prob_sm)
    recall_sm = recall_score(y_test, y_pred_sm, zero_division=0)
    prec_sm   = precision_score(y_test, y_pred_sm, zero_division=0)
    ap_sm     = average_precision_score(y_test, y_prob_sm)

    print('\n=== SMOTE + RF ===')
    print(f'AUC-ROC          : {auc_sm:.4f}')
    print(f'Avg Precision    : {ap_sm:.4f}')
    print(f'Recall  (fail)   : {recall_sm:.4f}')
    print(f'Precision (fail) : {prec_sm:.4f}')
    print()
    print(classification_report(y_test, y_pred_sm,
          target_names=['Delivered', 'Failed'], digits=4, zero_division=0))

    # ── Side-by-side comparison ──────────────────────────────────────────
    print('=== COMPARISON TABLE ===')
    comp = pd.DataFrame({
        'Metric': ['AUC-ROC', 'Recall (fail)', 'Precision (fail)', 'Avg Precision'],
        'Balanced Weight': [auc_base, recall_base, prec_base, ap_base],
        'SMOTE':           [auc_sm, recall_sm, prec_sm, ap_sm],
    })
    comp = comp.round(4)
    print(comp.to_string(index=False))
    best_model_name = 'SMOTE+RF' if recall_sm >= recall_base else 'Balanced-Weight RF'
    best_model      = rf_smote if recall_sm >= recall_base else rf_base
    best_probs      = y_prob_sm if recall_sm >= recall_base else y_prob_base
    print(f'\n→ Best model for recall: {best_model_name}')
else:
    print('SMOTE not available — using Balanced-Weight RF as best model')
    best_model_name = 'Balanced-Weight RF'
    best_model      = rf_base
    best_probs      = y_prob_base

---
## 6. Threshold Optimization

The default probability threshold (0.5) is designed for balanced classes. With a 140:1 imbalance, lowering the threshold towards the failure class improves recall at the cost of more false positives — an acceptable trade-off given the $17 cost of each missed failure vs a quick manual review.

In [ ]:
thresholds  = np.arange(0.05, 0.96, 0.05)
recall_list = []
prec_list   = []
f1_list     = []

for t in thresholds:
    y_t = (best_probs >= t).astype(int)
    recall_list.append(recall_score(y_test, y_t, zero_division=0))
    prec_list.append(precision_score(y_test, y_t, zero_division=0))
    f1_list.append(f1_score(y_test, y_t, zero_division=0))

thresh_df = pd.DataFrame({
    'threshold': thresholds,
    'recall':    recall_list,
    'precision': prec_list,
    'f1':        f1_list,
})

# Best threshold: maximize recall with precision > 0
valid = thresh_df[thresh_df['precision'] > 0]
best_thresh_row = valid.loc[valid['recall'].idxmax()]
best_thresh = best_thresh_row['threshold']

print('Threshold sweep (recall × precision × F1):')
print(thresh_df.round(4).to_string(index=False))
print(f'\n→ Optimal threshold for recall: {best_thresh:.2f}')
print(f'  Recall    : {best_thresh_row["recall"]:.4f}')
print(f'  Precision : {best_thresh_row["precision"]:.4f}')
print(f'  F1        : {best_thresh_row["f1"]:.4f}')

# ── Plot threshold curve ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, recall_list,    color=DANGER, lw=2.5, label='Recall')
ax.plot(thresholds, prec_list,      color=SAFE,   lw=2.5, label='Precision')
ax.plot(thresholds, f1_list,        color=ACCENT, lw=2,   label='F1', linestyle='--')
ax.axvline(best_thresh, color=WARN, linestyle=':', lw=2,
           label=f'Optimal threshold ({best_thresh:.2f})')
ax.set_xlabel('Decision Threshold', fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title(f'Threshold Optimization — {best_model_name}\n'
             'Lowering threshold trades precision for recall (preferred for $17 failure cost)',
             fontsize=12, fontweight='bold', color=NAVY)
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

Random Forest + SMOTE

In [ ]:
importances = pd.DataFrame({
    'feature':    FEATURES,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=True)

def imp_color(val):
    if val >= 0.20: return DANGER
    elif val >= 0.12: return WARN
    else: return ACCENT

bar_colors = [imp_color(v) for v in importances['importance']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(importances['feature'], importances['importance'],
               color=bar_colors, edgecolor='white', height=0.65)

for bar, val in zip(bars, importances['importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9, color=NAVY)

legend_patches = [
    mpatches.Patch(facecolor=DANGER, label='High importance (≥ 0.20)'),
    mpatches.Patch(facecolor=WARN,   label='Medium importance (0.12–0.20)'),
    mpatches.Patch(facecolor=ACCENT, label='Lower importance (< 0.12)'),
]
ax.legend(handles=legend_patches, fontsize=9)
ax.set_xlabel('Mean Decrease in Gini Impurity', fontweight='bold')
ax.set_title(
    f'Feature Importance — {best_model_name}\n'
    '(Model-based: mean decrease in impurity across 300 trees)',
    fontsize=12, fontweight='bold', color=NAVY
)
sns.despine(ax=ax)
plt.tight_layout()

out_path = FIGURES_DIR / 'feature_importance_final.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'  ✓ Saved → {out_path}')

print('\nFeature importances:')
print(importances.sort_values('importance', ascending=False).to_string(index=False))

---
## 8. ROC and Precision-Recall Curves

AUC-ROC summarizes discrimination across all thresholds. The Precision-Recall curve is more informative for imbalanced classes — it shows the recall vs precision trade-off at each threshold directly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── ROC curve ───────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, best_probs)
auc_val     = roc_auc_score(y_test, best_probs)
axes[0].plot(fpr, tpr, color=ACCENT, lw=2.5,
             label=f'ROC curve (AUC = {auc_val:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random classifier')
axes[0].fill_between(fpr, tpr, alpha=0.10, color=ACCENT)
axes[0].set_xlabel('False Positive Rate', fontweight='bold')
axes[0].set_ylabel('True Positive Rate', fontweight='bold')
axes[0].set_title(f'ROC Curve — {best_model_name}', fontweight='bold', color=NAVY)
axes[0].legend()
sns.despine(ax=axes[0])

# ── Precision-Recall curve ───────────────────────────────────────────────
prec_c, rec_c, _ = precision_recall_curve(y_test, best_probs)
ap_val           = average_precision_score(y_test, best_probs)
baseline_pr      = y_test.mean()
axes[1].plot(rec_c, prec_c, color=DANGER, lw=2.5,
             label=f'PR curve (AP = {ap_val:.4f})')
axes[1].axhline(baseline_pr, color='gray', linestyle='--', lw=1.2,
                label=f'Random classifier ({baseline_pr:.3f})')
axes[1].fill_between(rec_c, prec_c, alpha=0.10, color=DANGER)
axes[1].set_xlabel('Recall', fontweight='bold')
axes[1].set_ylabel('Precision', fontweight='bold')
axes[1].set_title(f'Precision-Recall Curve — {best_model_name}',
                  fontweight='bold', color=NAVY)
axes[1].legend()
sns.despine(ax=axes[1])

plt.suptitle('Model Evaluation — Imbalanced Class Performance',
             fontsize=13, fontweight='bold', color=NAVY)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Save Model Artifact

Saves the best model (and all supporting objects) to `ml/random_forest_model.pkl`.

In [ ]:
artifact = {
    'model':       best_model,
    'model_name':  best_model_name,
    'encoders':    encoders,
    'features':    FEATURES,
    'best_threshold': float(best_thresh),
    'metrics': {
        'auc_roc':          float(auc_val),
        'avg_precision':    float(ap_val),
        'recall_optimized': float(best_thresh_row['recall']),
        'precision_at_opt': float(best_thresh_row['precision']),
    },
    'dataset': {
        'source':    'Amazon LMRC 2021',
        'location':  'Los Angeles, CA',
        'period':    'July 2018',
        'n_packages': len(df),
        'n_failures':  int(df['delivery_failure'].sum()),
        'failure_rate': float(df['delivery_failure'].mean()),
    }
}

pkl_path = ML_DIR / 'random_forest_model.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(artifact, f)

print(f'Model saved → {pkl_path}')
print(f'File size   : {pkl_path.stat().st_size / 1024:.1f} KB')

# ── Verify round-trip ───────────────────────────────────────────────────
with open(pkl_path, 'rb') as f:
    check = pickle.load(f)

auc_check = roc_auc_score(y_test,
    check['model'].predict_proba(X_test)[:, 1])
print(f'Reload verify AUC : {auc_check:.4f}  ✓')

---
## 10. Summary of Findings

### Dataset
| Attribute | Value |
|-----------|-------|
| Source | Amazon LMRC 2021 (real data) |
| Geography | Los Angeles, CA |
| Period | July 2018 |
| Packages | 3,559 |
| Routes | 15 |
| Failure rate | 0.70% (25 failures) |
| Class imbalance | ~140:1 |

### Key Findings
| Finding | Value |
|---------|-------|
| carrier_D failure rate | 1.39% (highest) |
| carrier_B failure rate | 0.00% (zero failures) |
| Morning shift | 1.37% vs 0.55% afternoon |
| Routes < 40 km | 1.89% (urban density paradox) |
| Routes > 60 km | 0.00% |
| Zero-variance cols | days_in_fc, weather_risk |
| Data leakage col | delivery_failed (=target) |

### Modeling Decisions
- `class_weight='balanced'` corrects 140:1 imbalance
- SMOTE provides an additional comparison point
- Threshold tuned toward recall (cost: $17/missed failure)
- AUC-ROC and Average Precision are primary metrics

---
*Amazon LMRC 2021 (public dataset) — Correlation One DANA W12 — April 2026*